# WMS Colab Localizer

Web Media Studio 用の1セルLocalizerです。WMSでコピーしたYouTube URLを下のフォームへ貼り付け、権利確認にチェックして▶を押してください。

- Google Driveへの保存は不要です。
- ランタイムは一時VMです。生成した音声は処理後にブラウザへダウンロードします。
- 自分が権利を持つ、または保存・変換の許可を得ているコンテンツだけに使用してください。


In [ ]:
#@title WMS LOCALIZE — URLを貼って ▶ を押す { display-mode: "form" }
YOUTUBE_URL = "" #@param {type:"string"}
FORMAT = "mp3" #@param ["mp3", "m4a", "wav"]
MP3_BITRATE = "192" #@param ["128", "192", "256", "320"]
RIGHTS_CONFIRMED = False #@param {type:"boolean"}

import os
import pathlib
import shutil
import subprocess
import sys
import urllib.request
import zipfile

from google.colab import files

if not RIGHTS_CONFIRMED:
    raise RuntimeError("権利を持つ、または保存・変換の許可を得ているコンテンツであることを確認して、RIGHTS_CONFIRMED にチェックしてください。")

url = YOUTUBE_URL.strip()
if not url:
    raise RuntimeError("YOUTUBE_URL にURLまたは動画IDを貼り付けてください。")

print("[1/4] yt-dlp + EJS を準備しています…")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "yt-dlp[default]==2026.8.19"],
    check=True,
)

print("[2/4] Deno 2.9.6 を準備しています…")
deno_path = pathlib.Path("/usr/local/bin/deno")
needs_deno = True
if deno_path.exists():
    try:
        version = subprocess.check_output([str(deno_path), "--version"], text=True).splitlines()[0]
        needs_deno = "2.9.6" not in version
    except Exception:
        needs_deno = True

if needs_deno:
    archive = pathlib.Path("/tmp/deno-2.9.6.zip")
    urllib.request.urlretrieve(
        "https://github.com/denoland/deno/releases/download/v2.9.6/deno-x86_64-unknown-linux-gnu.zip",
        archive,
    )
    with zipfile.ZipFile(archive) as zf:
        zf.extract("deno", "/usr/local/bin")
    deno_path.chmod(0o755)

if shutil.which("ffmpeg") is None:
    print("FFmpegを準備しています…")
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True)

print("[3/4] 音声をLocalizeしています…")
output_dir = pathlib.Path("/content/wms-localizer-output")
shutil.rmtree(output_dir, ignore_errors=True)
output_dir.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, "-m", "yt_dlp",
    "--js-runtimes", "deno",
    "--no-playlist",
    "--match-filter", "duration <= 1800",
    "--extract-audio",
    "--audio-format", FORMAT,
    "--output", str(output_dir / "%(title).160B [%(id)s].%(ext)s"),
]
if FORMAT == "mp3":
    cmd.extend(["--audio-quality", f"{MP3_BITRATE}K"])
cmd.append(url)

subprocess.run(cmd, check=True)

candidates = [
    p for p in output_dir.iterdir()
    if p.is_file() and not p.name.endswith((".part", ".ytdl"))
]
if not candidates:
    raise RuntimeError("生成ファイルが見つかりませんでした。上のyt-dlpログを確認してください。")

output_file = max(candidates, key=lambda p: p.stat().st_mtime)
print(f"[4/4] 完了: {output_file.name} ({output_file.stat().st_size / 1024 / 1024:.1f} MB)")
print("ブラウザへのダウンロードを開始します。")
files.download(str(output_file))
